<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/6_Excercise_Explicit_Memory_Spaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# thrust::copy(source_vector.begin(), source_vector.end(), dst_vector.begin());

In [1]:
import os

if os.getenv("COLAB_RELEASE_TAG"): # If running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.06-Memory-Spaces/Sources/ach.h -nv -O Sources/ach.h

2026-09-26 21:05:09 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.06-Memory-Spaces/Sources/ach.h [3296/3296] -> "Sources/ach.h" [1]


**Original Code**

In [3]:
%%writefile Sources/heat-2D-implicit-memory-spaces.cpp
#include "ach.h"

int main()
{
  int height = 4096;
  int width  = 4096;

  thrust::universal_vector<float> prev = ach::init(height, width);
  thrust::universal_vector<float> next(height * width);

  for (int write_step = 0; write_step < 3; write_step++) {
    std::printf("   write step %d\n", write_step);
    ach::store(write_step, height, width, prev);

    for (int compute_step = 0; compute_step < 3; compute_step++) {
      auto begin = std::chrono::high_resolution_clock::now();
      ach::simulate(height, width, prev, next);
      auto end = std::chrono::high_resolution_clock::now();
      auto seconds = std::chrono::duration<double>(end - begin).count();
      std::printf("computed step %d in %g s\n", compute_step, seconds);
      prev.swap(next);
    }
  }
}

Writing Sources/heat-2D-implicit-memory-spaces.cpp


In [4]:
!nvcc --extended-lambda -o /tmp/a.out Sources/heat-2D-implicit-memory-spaces.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

   write step 0
computed step 0 in 0.0225391 s
computed step 1 in 0.000634825 s
computed step 2 in 0.000598141 s
   write step 1
computed step 0 in 0.0269236 s
computed step 1 in 0.000638346 s
computed step 2 in 0.000598007 s
   write step 2
computed step 0 in 0.0390473 s
computed step 1 in 0.00064012 s
computed step 2 in 0.000609587 s


**use memory spaces and explicity copy data to the disc**

In [6]:
%%writefile Sources/heat-2D-explicit-memory-spaces.cpp
#include "ach.h"

int main()
{
  int height = 4096;
  int width  = 4096;

  // Use explicit memory space containers

  thrust::device_vector<float> d_prev = ach::init(height, width);
  thrust::device_vector<float> d_next(height*width);
  thrust::host_vector<float> h_prev(height * width);

  for (int write_step = 0; write_step < 3; write_step++) {
    std::printf("   write step %d\n", write_step);
    thrust::copy(d_prev.begin(), d_prev.end(), h_prev.begin());
    ach::store(write_step, height, width, h_prev);

    for (int compute_step = 0; compute_step < 3; compute_step++) {
      auto begin = std::chrono::high_resolution_clock::now();
      ach::simulate(height, width, d_prev, d_next);
      auto end = std::chrono::high_resolution_clock::now();
      auto seconds = std::chrono::duration<double>(end - begin).count();
      std::printf("computed step %d in %g s\n", compute_step, seconds);
      d_prev.swap(d_next);
    }
  }
}

Overwriting Sources/heat-2D-explicit-memory-spaces.cpp


In [7]:
!nvcc --extended-lambda -o /tmp/a.out Sources/heat-2D-explicit-memory-spaces.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

   write step 0
computed step 0 in 0.000852913 s
computed step 1 in 0.000696649 s
computed step 2 in 0.000690878 s
   write step 1
computed step 0 in 0.000756886 s
computed step 1 in 0.000723811 s
computed step 2 in 0.000722753 s
   write step 2
computed step 0 in 0.000811049 s
computed step 1 in 0.000775901 s
computed step 2 in 0.000772192 s
